# Titanic Dataset Analysis: Predictive Model Development

## Overview
This notebook implements a complete ML pipeline for predicting Titanic passenger survival through:
1. **Data Cleaning** - Handling missing values and outliers
2. **Feature Engineering** - Creating derived features
3. **Feature Selection** - Identifying the most predictive features
4. **Exploratory Analysis** - Visualizing patterns and relationships

---

## 1. Imports & Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add scripts directory to path
notebook_dir = Path('__file__').parent.absolute()
scripts_dir = notebook_dir.parent / 'scripts'
sys.path.insert(0, str(scripts_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Import our custom modules
from data_cleaning import load_data, clean_data, identify_missing_values
from feature_engineering import engineer_features
from feature_selection import select_features

print('✓ All imports successful')

## 2. Initial Data Exploration

In [ ]:
# Load raw data
data_dir = notebook_dir.parent / 'data'
train_path = data_dir / 'train.csv'

df_raw = pd.read_csv(train_path)

print('DATASET OVERVIEW')
print('=' * 70)
print(f'Shape: {df_raw.shape}')
print(f'\nFirst 5 rows:')
print(df_raw.head())
print(f'\nData types:')
print(df_raw.dtypes)
print(f'\nBasic statistics:')
print(df_raw.describe())

### 2.1 Missing Values Analysis

In [ ]:
# Analyze missing values
missing_info = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage': (df_raw.isnull().sum() / len(df_raw)) * 100
}).sort_values('Missing_Count', ascending=False)

missing_info = missing_info[missing_info['Missing_Count'] > 0]
print('MISSING VALUES ANALYSIS')
print('=' * 70)
print(missing_info.to_string(index=False))

# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 6))
missing_info_sorted = missing_info.sort_values('Missing_Percentage')
ax.barh(missing_info_sorted['Column'], missing_info_sorted['Missing_Percentage'], color='coral')
ax.set_xlabel('Missing Percentage (%)', fontsize=12)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey Observations:')
print('- Age: 19.9% missing (important feature, impute wisely)')
print('- Cabin: 77.1% missing (too sparse, convert to binary indicator)')
print('- Embarked: 0.2% missing (minimal, use mode)')

### 2.2 Target Variable Distribution

In [ ]:
# Analyze target variable
survival_counts = df_raw['Survived'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
axes[0].bar(['Died (0)', 'Survived (1)'], survival_counts.values, color=['#d62728', '#2ca02c'])
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Survival Distribution', fontsize=12, fontweight='bold')
for i, v in enumerate(survival_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
colors = ['#d62728', '#2ca02c']
axes[1].pie(survival_counts.values, labels=['Died (0)', 'Survived (1)'], autopct='%1.1f%%',
             colors=colors, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title('Survival Rate', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Total Passengers: {len(df_raw)}')
print(f'Survived: {survival_counts[1]} ({survival_counts[1]/len(df_raw)*100:.1f}%)')
print(f'Died: {survival_counts[0]} ({survival_counts[0]/len(df_raw)*100:.1f}%)')

## 3. Data Cleaning Pipeline

In [ ]:
# Run data cleaning
cleaned_path = data_dir / 'train_cleaned.csv'
df_cleaned, stats = clean_data(str(train_path), str(cleaned_path))

print('\nCleaned data statistics:')
print(df_cleaned.describe())

### 3.1 Cleaning Decisions & Justification

In [ ]:
cleaning_decisions = pd.DataFrame({
    'Issue': [
        'Missing Age (19.9%)',
        'Missing Embarked (0.2%)',
        'Missing Fare (0.1%)',
        'Missing Cabin (77.1%)',
        'Fare Outliers',
        'Age Outliers'
    ],
    'Strategy': [
        'Impute with median by Pclass',
        'Impute with mode',
        'Impute with median by Pclass',
        'Convert to HasCabin binary',
        'Cap using IQR method',
        'Clip to [0, 120]'
    ],
    'Justification': [
        'Preserves class distributions; class affects age at booking',
        'Minimal missing data; mode is reasonable',
        'Preserves class structure; more nuanced than overall median',
        'Too sparse to impute; binary indicator captures cabin knowledge',
        'Preserves data while limiting outlier influence on models',
        'Age is naturally bounded; prevents unrealistic values'
    ]
})

print('DATA CLEANING DECISIONS')
print('=' * 100)
for idx, row in cleaning_decisions.iterrows():
    print(f"\n{idx+1}. {row['Issue']}")
    print(f"   Strategy: {row['Strategy']}")
    print(f"   Justification: {row['Justification']}")

## 4. Feature Engineering Pipeline

In [ ]:
# Run feature engineering
engineered_path = data_dir / 'train_engineered.csv'
df_engineered = engineer_features(str(cleaned_path), str(engineered_path))

print(f'\nFeature count increased from {len(df_cleaned.columns)} to {len(df_engineered.columns)}')

### 4.1 Family Features Analysis

In [ ]:
print('FAMILY FEATURES')
print('=' * 70)
print('\nFamilySize Distribution:')
print(df_engineered['FamilySize'].value_counts().sort_index())

# Survival by FamilySize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Family size vs survival
family_survival = df_engineered.groupby('FamilySize')['Survived'].agg(['sum', 'count'])
family_survival['survival_rate'] = family_survival['sum'] / family_survival['count']

axes[0].bar(family_survival.index, family_survival['survival_rate'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Family Size', fontsize=11)
axes[0].set_ylabel('Survival Rate', fontsize=11)
axes[0].set_title('Survival Rate by Family Size', fontsize=12, fontweight='bold')
axes[0].axhline(y=df_engineered['Survived'].mean(), color='red', linestyle='--', label='Overall mean')
axes[0].legend()
axes[0].set_ylim(0, 1)

# IsAlone vs Survival
alone_counts = df_engineered['IsAlone'].value_counts()
alone_survival = df_engineered.groupby('IsAlone')['Survived'].value_counts().unstack()

alone_survival.plot(kind='bar', ax=axes[1], color=['#d62728', '#2ca02c'])
axes[1].set_xlabel('Is Alone', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Survival Distribution by IsAlone', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['Not Alone (0)', 'Alone (1)'], rotation=0)
axes[1].legend(['Died', 'Survived'])

plt.tight_layout()
plt.show()

print('\nKey Insight: Traveling alone correlates with lower survival rates')

### 4.2 Title & Name Features

In [ ]:
print('TITLE FEATURES')
print('=' * 70)
print('\nTitle Distribution:')
print(df_engineered['Title'].value_counts())

# Title survival rates
title_survival = df_engineered.groupby('Title')['Survived'].agg(['sum', 'count', 'mean'])
title_survival.columns = ['Survived', 'Total', 'SurvivalRate']
title_survival = title_survival.sort_values('SurvivalRate', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(title_survival.index, title_survival['SurvivalRate'], color='teal', alpha=0.7)
ax.set_xlabel('Survival Rate', fontsize=11)
ax.set_title('Survival Rate by Passenger Title', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
for i, v in enumerate(title_survival['SurvivalRate']):
    ax.text(v + 0.02, i, f'{v:.1%}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(title_survival.to_string())
print('\nKey Insight: Female titles (Mrs, Miss) had much higher survival; Male titles much lower')

### 4.3 Age Groups Analysis

In [ ]:
print('AGE GROUP FEATURES')
print('=' * 70)
print('\nAge Group Distribution:')
print(df_engineered['AgeGroup'].value_counts().sort_index())

# Age group survival
age_group_survival = df_engineered.groupby('AgeGroup')['Survived'].agg(['sum', 'count', 'mean'])
age_group_survival.columns = ['Survived', 'Total', 'SurvivalRate']

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']
ax.bar(range(len(age_group_survival)), age_group_survival['SurvivalRate'], color=colors, alpha=0.7)
ax.set_xticks(range(len(age_group_survival)))
ax.set_xticklabels(age_group_survival.index)
ax.set_ylabel('Survival Rate', fontsize=11)
ax.set_title('Survival Rate by Age Group', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.axhline(y=df_engineered['Survived'].mean(), color='red', linestyle='--', label='Overall mean')
ax.legend()

for i, v in enumerate(age_group_survival['SurvivalRate']):
    ax.text(i, v + 0.03, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(age_group_survival.to_string())
print('\nKey Insight: Children had highest survival; adults and seniors lower')

### 4.4 Fare Features & Transformations

In [ ]:
print('FARE FEATURES & TRANSFORMATIONS')
print('=' * 70)

# Visualize log transformation effect
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original Fare distribution
axes[0, 0].hist(df_engineered['Fare'].dropna(), bins=30, color='coral', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Original Fare Distribution (Right-Skewed)', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Fare')
axes[0, 0].set_ylabel('Frequency')

# Log transformed
axes[0, 1].hist(df_engineered['LogFare'].dropna(), bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Log-Transformed Fare (Normalized)', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Log(Fare)')
axes[0, 1].set_ylabel('Frequency')

# Fare per person
axes[1, 0].scatter(df_engineered['Fare'], df_engineered['Survived'], alpha=0.4)
axes[1, 0].set_title('Fare vs Survival', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Fare')
axes[1, 0].set_ylabel('Survived')

# Fare category
fare_survival = df_engineered.groupby('FareCategory', observed=True)['Survived'].value_counts().unstack()
fare_survival.plot(kind='bar', ax=axes[1, 1], color=['#d62728', '#2ca02c'])
axes[1, 1].set_title('Survival by Fare Category', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Fare Category')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45)
axes[1, 1].legend(['Died', 'Survived'])

plt.tight_layout()
plt.show()

print('\nKey Insight: Higher fare → higher survival (paid more = higher class/priority)')

## 5. Feature Selection Pipeline

In [ ]:
# Run feature selection
final_path = data_dir / 'train_final.csv'
selected_features = select_features(str(engineered_path), str(final_path), n_features=20)

### 5.1 Correlation Analysis

In [ ]:
# Calculate correlations
numerical_cols = df_engineered.select_dtypes(include=[np.number]).columns
corr_matrix = df_engineered[numerical_cols].corr()

# Correlation with target
target_corr = corr_matrix['Survived'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
target_corr_filtered = target_corr[target_corr.index != 'Survived'].head(20)
colors = ['green' if x > 0 else 'red' for x in target_corr_filtered.values]
ax.barh(range(len(target_corr_filtered)), target_corr_filtered.values, color=colors, alpha=0.7)
ax.set_yticks(range(len(target_corr_filtered)))
ax.set_yticklabels(target_corr_filtered.index)
ax.set_xlabel('Correlation with Survived', fontsize=11)
ax.set_title('Top 20 Features by Correlation with Target', fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

for i, v in enumerate(target_corr_filtered.values):
    ax.text(v + 0.01 if v > 0 else v - 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('Top 15 Features by Correlation with Survival:')
print(target_corr[target_corr.index != 'Survived'].head(15).to_string())

### 5.2 Feature Importance Ranking

In [ ]:
# Train quick Random Forest for feature importance
X = df_engineered.drop(columns=['PassengerId', 'Name', 'Ticket', 'Sex', 'Embarked', 'Title', 'AgeGroup', 'FareCategory', 'Survived'], errors='ignore')
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.mean())
y = df_engineered['Survived']

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top_features = feature_importance.head(20)
ax.barh(range(len(top_features)), top_features['Importance'].values, color='steelblue', alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_title('Top 20 Features by Random Forest Importance', fontsize=12, fontweight='bold')
ax.invert_yaxis()

for i, v in enumerate(top_features['Importance'].values):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('Top 20 Features by Importance:')
print(feature_importance.head(20).to_string(index=False))

## 6. Summary & Conclusions

In [ ]:
print('\n' + '=' * 80)
print('TITANIC ANALYSIS - COMPLETE SUMMARY')
print('=' * 80)

print('\n1. DATA CLEANING RESULTS:')
print(f'   - Initial rows: 891')
print(f'   - Final rows: {len(df_cleaned)} (100% retained)')
print(f'   - Missing values handled: Age, Embarked, Fare, Cabin')
print(f'   - Outliers handled: Fare (IQR method), Age (clipping)')
print(f'   - Final missing values: 0')

print('\n2. FEATURE ENGINEERING RESULTS:')
print(f'   - Initial features: {len(df_cleaned.columns)}')
print(f'   - Engineered features: {len(df_engineered.columns)}')
print(f'   - Feature categories:')
print(f'     * Family features: 2 (FamilySize, IsAlone)')
print(f'     * Title features: Multiple (One-hot encoded)')
print(f'     * Age features: 2 (AgeGroup bins + continuous)')
print(f'     * Fare features: 4 (Fare, LogFare, FarePerPerson, FareCategory)')
print(f'     * Interaction features: 5 (Age×Title, Fare×Pclass, etc.)')
print(f'     * Encoded categorical: 10+ (Sex, Embarked, Title, Pclass)')

print('\n3. FEATURE SELECTION RESULTS:')
print(f'   - Features selected: 20 (from 40+ engineered)')
print(f'   - Selection criteria: Importance rank, Correlation, Multicollinearity')
print(f'   - Most predictive: Title, Fare, Sex, Age-based features')

print('\n4. KEY SURVIVAL PREDICTORS:')
print(f'   Strong (|r| > 0.4): Sex, Fare, Title, Pclass')
print(f'   Moderate (0.2 < |r| < 0.4): Age, FamilySize, IsAlone')
print(f'   Weak (|r| < 0.2): Other engineered features')

print('\n5. DATA READY FOR MODELING:')
print(f'   ✓ No missing values')
print(f'   ✓ Outliers handled')
print(f'   ✓ Features engineered & selected')
print(f'   ✓ Ready for: Logistic Regression, Random Forest, XGBoost, etc.')

print('\n' + '=' * 80)